# Machine Learning: The Big Picture

Over the last few days the LMS has thrown a lot at you: predictive analytics, supervised and unsupervised learning, linear and logistic regression, advanced ML methods, image analysis, text analysis. Each piece made sense on its own, probably. The trouble is that they arrive as a list, and a list isn't a mental model.

This notebook is the map. By the end you should be able to look at any new ML problem and answer three questions:

1. **What kind of learning is this?** Supervised, unsupervised, or reinforcement.
2. **What kind of answer am I after?** A number, or a category.
3. **How would I know if my model is any good?** Which is harder than it sounds.

The first three parts are for reading and running, the explanations are the point. The last part is a challenge where you build a working classifier yourself.

We'll use the **Titanic** dataset throughout, because you already know it. That's deliberate: when you're learning a new idea, it helps enormously if the data isn't also new.

**What you need:** `titanic.csv` in the same folder, plus pandas, scikit-learn and matplotlib.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('titanic.csv')
print(df.shape)
df.head()

(891, 8)


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


---
# Part 1 , What machine learning actually is

Here's the whole idea, and it's smaller than the hype suggests.

**Normal programming:** you write the rules, the computer applies them to data.

    if sex == 'female' and pclass == 1:
        prediction = 'survived'

You worked out that rule yourself. You looked at the data, you spotted a pattern, you wrote it down.

**Machine learning:** you give the computer the data *and* the answers, and it works out the rules.

Nobody tells the algorithm that women were more likely to survive. It finds that out, from 891 examples, along with a lot of other patterns you might not have spotted, and it weighs them all against each other.

That's the trade. You give up writing the rules, and in return you get rules you couldn't have written by hand. You also give up knowing exactly why the model says what it says, which matters enormously in a job where somebody is going to ask you exactly that.

In [2]:
# The data we're learning from: 891 passengers, and we know what happened to each.
print("Passengers:", len(df))
print("Survived:  ", (df['survived'] == 1).sum())
print("Died:      ", (df['survived'] == 0).sum())
print()
print("Overall survival rate: {:.1%}".format(df['survived'].mean()))

Passengers: 891
Survived:   342
Died:       549

Overall survival rate: 38.4%


### The patterns are really there

Before any model, look at what the data already tells you. These are the patterns an algorithm would have to discover on its own.

In [3]:
print("Survival rate by sex:")
print(df.groupby('sex')['survived'].mean().round(4))
print()
print("Survival rate by passenger class:")
print(df.groupby('pclass')['survived'].mean().round(4))

Survival rate by sex:
sex
female    0.7420
male      0.1889
Name: survived, dtype: float64

Survival rate by passenger class:
pclass
1    0.6296
2    0.4728
3    0.2424
Name: survived, dtype: float64


Women survived at **74.2%**, men at **18.9%**. First class **63.0%**, third class **24.2%**.

Those two patterns are strong and obvious. A model's job is to find them, combine them, and handle the cases where they conflict, like a man in first class or a woman in third.

> 💡 **Worth noticing.** You just did the most important step in any ML project, and you did it with `groupby`. Understanding your data before modelling isn't a warm-up, it's how you'll later tell whether the model is sensible or nonsense.

---
# Part 2 , The three kinds of learning

Every ML problem you meet falls into one of three families. Getting this right is the first decision you make, because it determines everything after it.

## Supervised learning , you have the answers

You have historical data **with the answer attached**. Titanic is supervised: every row has a `survived` column telling you what actually happened. The algorithm learns by comparing its guesses to the real answers and adjusting.

This splits into two, depending on what kind of answer you want:

- **Classification** predicts a *category*. Did this passenger survive, yes or no? Is this email spam? Which of five products will this customer buy?
- **Regression** predicts a *number*. What fare did they pay? What will this house sell for? How many units will we ship next month?

The names are unhelpful, sorry. **Logistic regression is a classification algorithm**, despite the name. Everyone finds this confusing, and it stays confusing.

## Unsupervised learning , no answers, find structure

You have data with **no labels**, and you want the algorithm to find structure in it. Nobody tells it what's right, because there's no "right".

On Titanic, that would be: *forget the survived column entirely, and group these passengers into clusters based on how similar they are.* You might get something like "wealthy older couples", "young men travelling alone", "families in third class". The algorithm doesn't name those groups, it just finds them, and interpreting them is your job.

That's K-Means, which is coming up on Monday.

## Reinforcement learning , learn by trial and error

An agent takes actions in an environment and gets rewards or penalties, and over many attempts it learns which actions pay off. This is how systems learn to play games, drive cars and control robots.

There's no fixed dataset here, the agent generates its own experience by trying things. It's fascinating, it's genuinely different from the other two, and you will almost certainly never use it in a data analytics job. Know what it is, know why it doesn't apply to a spreadsheet of customers, and move on.

## The one-line version

| | You have | You want | Titanic example |
|---|---|---|---|
| **Supervised** | Data + answers | Predict the answer for new data | Will this passenger survive? |
| **Unsupervised** | Data only | Find structure | What types of passenger were there? |
| **Reinforcement** | An environment | A good strategy | Not applicable, no environment to act in |

> ⚠️ **The question that decides it:** *do I have labelled examples of the thing I'm trying to predict?* If yes, supervised. If no, unsupervised. Ask this before you write a single line of modelling code.

---
# Part 3 , Linear vs logistic regression

These two get taught together and confused constantly, so let's separate them properly.

**Linear regression predicts a number.** You fit a straight line through your data and read values off it. Given a passenger's class and age, what fare did they pay? The answer is a number and could be anything.

**Logistic regression predicts a probability**, which you then turn into a category. Given a passenger's class and age, what's the chance they survived? The answer must sit between 0 and 1.

That constraint is the entire reason logistic regression exists.

### Why not just use a straight line for yes/no?

Reasonable question. `survived` is already 0 or 1, so why not fit a line to it and call anything above 0.5 a survivor?

Let's actually try it and see what breaks.

In [4]:
from sklearn.linear_model import LinearRegression, LogisticRegression

# Use fare to predict survival, with a straight line.
d = df.dropna(subset=['age']).copy()
X = d[['fare']].values
y = d['survived'].values

linear = LinearRegression().fit(X, y)

print("A straight line's predictions:")
for fare in [0, 50, 200, 300, 512]:
    p = linear.predict([[fare]])[0]
    flag = "   <-- IMPOSSIBLE" if p < 0 or p > 1 else ""
    print(f"  fare £{fare:<4}  ->  {p:+.4f}{flag}")

A straight line's predictions:
  fare £0     ->  +0.3197
  fare £50    ->  +0.4443
  fare £200   ->  +0.8179
  fare £300   ->  +1.0670   <-- IMPOSSIBLE
  fare £512   ->  +1.5950   <-- IMPOSSIBLE


There it is. The line says a passenger who paid £512 has a **1.5958** chance of survival, which is a 159% probability. That is not a thing.

It isn't only theoretical, either. Three real passengers in this dataset get impossible predictions. The problem is structural: a straight line goes on forever in both directions, and probabilities don't.

In [5]:
preds = linear.predict(X)
print("Predictions above 1 (impossible):", (preds > 1).sum())
print("Highest prediction: {:.4f}".format(preds.max()))

Predictions above 1 (impossible): 3
Highest prediction: 1.5958


### What logistic regression does instead

It takes that straight line and bends it through a function that squashes any number, however large or small, into the range 0 to 1. The result is an **S-shaped curve** (a sigmoid).

Very negative inputs approach 0, very positive inputs approach 1, and it never crosses either. Same data, same idea, but the output is always a valid probability.

In [6]:
logistic = LogisticRegression().fit(X, y)

print("Logistic regression on the same data:")
for fare in [0, 50, 200, 300, 512]:
    p = logistic.predict_proba([[fare]])[0, 1]
    print(f"  fare £{fare:<4}  ->  P(survive) = {p:.4f}")

probs = logistic.predict_proba(X)[:, 1]
print()
print("Range of all predictions: {:.4f} to {:.4f}".format(probs.min(), probs.max()))
print("Always between 0 and 1, by construction.")

Logistic regression on the same data:
  fare £0     ->  P(survive) = 0.2897
  fare £50    ->  P(survive) = 0.4758
  fare £200   ->  P(survive) = 0.9091
  fare £300   ->  P(survive) = 0.9802
  fare £512   ->  P(survive) = 0.9993

Range of all predictions: 0.2897 to 0.9993
Always between 0 and 1, by construction.


### The two side by side

One picture makes this stick better than any explanation.

In [7]:
fare_grid = np.linspace(0, 550, 300).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(d['fare'], d['survived'], alpha=0.15, label='actual passengers')
ax.plot(fare_grid, linear.predict(fare_grid), 'r--', lw=2,
        label='linear (straight line)')
ax.plot(fare_grid, logistic.predict_proba(fare_grid)[:, 1], 'g-', lw=2,
        label='logistic (S-curve)')
ax.axhline(1, color='grey', ls=':', lw=1)
ax.axhline(0, color='grey', ls=':', lw=1)
ax.set_xlabel('Fare paid (£)')
ax.set_ylabel('Survived (0 or 1) / probability')
ax.set_title('Why classification needs a curve, not a line')
ax.legend()
plt.show()

<Figure size 900x500 with 1 Axes>

The dotted grey lines mark 0 and 1, the only values a probability is allowed to take.

The **red dashed line** sails straight past 1 and keeps going. The **green curve** flattens out and approaches 1 without ever reaching it, which is exactly what you want when the answer is "almost certainly yes, but never guaranteed".

> 💡 **The rule of thumb.** Predicting a number, use linear regression. Predicting a category, use logistic regression. The word "regression" in both names is a historical accident, don't read anything into it.

---
---
# Part 4 , Your turn: build a survival classifier

Enough reading. Below you'll build a working model, in the order a real project actually goes.

The order matters more than the code, and it's the bit most beginners skip:

1. Establish a **baseline** first, so you know what "good" even means
2. Try the **simplest possible rule**, so you know whether a model is earning its keep
3. Only then **train a model**, and compare it honestly against both

Fill in each `# --- YOUR CODE ---` block. All the solutions are at the very bottom under a `SOLUTIONS` banner, so have a real go first.

## Stage 1 , The baseline

Before any model, ask: what accuracy do I get by being as stupid as possible? Every model must beat this, and it's shocking how often one doesn't.

The laziest possible classifier predicts the same thing for everyone. Since 549 of 891 passengers died, predicting "died" for every single person gets you a certain accuracy for free.

In [8]:
# Lesson:
# A baseline is the score to beat. Without one, "my model is 80% accurate"
# is a meaningless sentence.

# --- YOUR CODE ---
# Task 1.1: work out the accuracy of predicting that EVERYONE died.
# (hint: it's just the proportion of passengers who actually died)




## Stage 2 , The simplest rule

You already know from Part 1 that women survived at 74.2% and men at 18.9%. So here's a rule that needs no machine learning at all: **predict every woman survives, and every man dies.**

Score it. This is the number a model genuinely has to beat.

In [9]:
# Lesson:
# Comparing a boolean Series to the truth gives you accuracy directly, because
# True counts as 1: (predictions == actual).mean()

# --- YOUR CODE ---
# Task 2.1: build predictions using ONLY sex , 1 if female, 0 if male.
# Then compare against df['survived'] and print the accuracy.




> 💡 That simple rule should get you comfortably into the high seventies. Sit with that for a second: no model, no training, no scikit-learn, one line of logic. Any model you build now has to beat it or it isn't worth deploying.

## Stage 3 , Prepare the data

Models need numbers. Two things are in the way:

- `sex` is text, so it needs encoding into 0 and 1
- `age` has 177 missing values, and scikit-learn refuses to train with gaps

You met both of these in the feature-engine session, so this is revision.

In [10]:
# Lesson:
# Encode a two-value text column with a comparison:  (df['col'] == 'value').astype(int)
# Fill missing numbers with the median, which is safer than the mean when the
# column is skewed.

# --- YOUR CODE ---
# Task 3.1: make a copy of df called `data`.
# Task 3.2: fill missing 'age' with the median age.
# Task 3.3: create a 'sex_male' column: 1 for male, 0 for female.
# Task 3.4: confirm there are no missing values left in the columns you need.




## Stage 4 , Split into training and test sets

**This is the step that separates real modelling from self-deception.**

If you train a model on all 891 passengers and then score it on those same 891, you learn nothing. The model has seen every answer. It's an exam where the student had the mark scheme.

So you hold some data back. Train on most of it, then test on data the model has never seen. That test score is the only honest estimate of how it performs on new passengers.

In [ ]:
# Lesson:
# train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
#   test_size    fraction held back for testing
#   random_state fixes the shuffle so your results are reproducible
#   stratify=y   keeps the survived/died ratio the same in both halves

# --- YOUR CODE ---
# Task 4.1: import train_test_split from sklearn.model_selection.
# Task 4.2: build X from ['pclass','sex_male','age','sibsp','parch','fare']
#           and y from 'survived'.
# Task 4.3: split with test_size=0.2, random_state=42, stratify=y.
# Task 4.4: print how many rows ended up in each set.
import statsmodels.formula.api as smf

ols_model = smf.ols('survived ~ fare*sex_male', data=data).fit()

print(ols_model.summary())


> ⚠️ **Never skip `random_state`.** Without it the split is different every run, your accuracy changes each time, and you can't tell whether a change you made helped or whether you just got a luckier shuffle.

## Stage 5 , Train the model

The part everyone thinks is the whole job. It's three lines.

Every scikit-learn model works the same way: create it, `.fit()` it on the training data, `.predict()` with it. Swap `LogisticRegression` for a decision tree or a random forest and the rest of your code doesn't change.

In [12]:
# Lesson:
# model = LogisticRegression(max_iter=1000)
# model.fit(X_train, y_train)
# predictions = model.predict(X_test)
#
# max_iter=1000 just gives it more attempts to settle; the default sometimes
# warns that it stopped early.

# --- YOUR CODE ---
# Task 5.1: create a LogisticRegression with max_iter=1000.
# Task 5.2: fit it on the TRAINING data only.
# Task 5.3: predict on the TEST data.




## Stage 6 , Score it honestly

Now compare against your baseline and your simple rule. That comparison is the actual result, not the accuracy on its own.

In [ ]:
# Lesson:
# from sklearn.metrics import accuracy_score
# accuracy_score(y_test, predictions)

# --- YOUR CODE ---
# Task 6.1: print the test accuracy.
# Task 6.2: print it next to the baseline (Stage 1) and the rule (Stage 2).
# Was the model worth the effort?

from sklearn.metrics import accuracy_score




## Stage 7 , Ask what it learned

A model you can't explain is a liability, and "the algorithm decided" is not an answer you can give a manager.

Logistic regression is unusually friendly here, because you can read its coefficients. A **positive** coefficient pushes the prediction towards survival, a **negative** one pushes away, and the size tells you how strongly.

In [14]:
# Lesson:
# model.coef_[0] holds one coefficient per feature, in the same order as
# your X columns. Zip them together to read them.

# --- YOUR CODE ---
# Task 7.1: print each feature name alongside its coefficient.
# Task 7.2: which feature matters most? Does the direction match what you
#           saw in Part 1?




> 💡 **The sanity check.** If a coefficient's direction surprises you, that's a signal, not a curiosity. It usually means a data problem, a leaked column, or a feature you've misunderstood. Reading coefficients is how you catch a broken model before it embarrasses you.

## Stage 8 (bonus) , Predict for a new passenger

The point of a trained model is applying it to someone it's never seen. Invent a passenger and ask for the *probability*, not just the yes/no.

In [15]:
# Lesson:
# model.predict_proba(X_new) returns [[P(died), P(survived)]]
# so [0, 1] gets you the survival probability.
# Build X_new as a DataFrame with the SAME columns in the SAME order as X.

# --- YOUR CODE ---
# Task 8.1: predict the survival probability for a 25-year-old woman in
#           1st class who paid £80, travelling alone.
# Task 8.2: do the same for a 30-year-old man in 3rd class who paid £8.
# Compare the two numbers.




---
---
# SOLUTIONS

Have a genuine go at each stage first. The code is short; the thinking about
baselines, splits and coefficients is the part that's worth your time.

### Stage 1 , The baseline

In [16]:
baseline = (df['survived'] == 0).mean()
print("Predict everyone died: {:.4f}".format(baseline))   # 0.6162
# 61.6% accurate while knowing nothing whatsoever about anybody.

Predict everyone died: 0.6162


### Stage 2 , The simplest rule

In [17]:
rule_predictions = (df['sex'] == 'female').astype(int)
rule_accuracy = (rule_predictions == df['survived']).mean()
print("Women live, men die: {:.4f}".format(rule_accuracy))   # 0.7868
# 78.7% from a single line of logic and no machine learning at all.

Women live, men die: 0.7868


### Stage 3 , Prepare the data

In [18]:
data = df.copy()
data['age'] = data['age'].fillna(data['age'].median())     # median age is 28.0
data['sex_male'] = (data['sex'] == 'male').astype(int)

print(data[['pclass', 'sex_male', 'age', 'sibsp', 'parch', 'fare', 'survived']].isnull().sum())
# all zeros , nothing missing in the columns we're about to use

pclass      0
sex_male    0
age         0
sibsp       0
parch       0
fare        0
survived    0
dtype: int64


### Stage 4 , Train/test split

In [19]:
from sklearn.model_selection import train_test_split

features = ['pclass', 'sex_male', 'age', 'sibsp', 'parch', 'fare']
X = data[features]
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))   # 712
print("Test rows:    ", len(X_test))    # 179

Training rows: 712
Test rows:     179


### Stage 5 , Train the model

In [20]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Trained. Predictions made for", len(predictions), "test passengers.")

Trained. Predictions made for 179 test passengers.


### Stage 6 , Score it honestly

In [21]:
from sklearn.metrics import accuracy_score

model_accuracy = accuracy_score(y_test, predictions)

print("Baseline (everyone died): {:.4f}".format(baseline))          # 0.6162
print("Simple rule (sex only):   {:.4f}".format(rule_accuracy))     # 0.7868
print("Logistic regression:      {:.4f}".format(model_accuracy))    # 0.8045

# The model wins, but by under two percentage points over one line of logic.
# That is a completely normal and very healthy result to get. It tells you the
# sex pattern was carrying most of the signal all along, and it's exactly the
# kind of honesty that stops people overselling models.

Baseline (everyone died): 0.6162
Simple rule (sex only):   0.7868
Logistic regression:      0.8045


### Stage 7 , Ask what it learned

In [22]:
for name, coef in zip(features, model.coef_[0]):
    print(f"  {name:10} {coef:+.4f}")

# pclass     -1.0514   higher class number (3rd) = much less likely to survive
# sex_male   -2.6126   being male is by far the strongest effect, and negative
# age        -0.0385   older slightly less likely
# sibsp      -0.2673   more siblings/spouses aboard, slightly less likely
# parch      -0.1006   more parents/children aboard, slightly less likely
# fare       +0.0032   paying more, very slightly more likely

# sex_male dominates, and it's negative. That matches Part 1 exactly:
# women 74.2%, men 18.9%. The model found the same pattern you did.

  pclass     -1.0515
  sex_male   -2.6130
  age        -0.0385
  sibsp      -0.2672
  parch      -0.1009
  fare       +0.0032


### Stage 8 , Predict for a new passenger

In [23]:
new_passengers = pd.DataFrame([
    {'pclass': 1, 'sex_male': 0, 'age': 25, 'sibsp': 0, 'parch': 0, 'fare': 80},
    {'pclass': 3, 'sex_male': 1, 'age': 30, 'sibsp': 0, 'parch': 0, 'fare': 8},
], columns=features)

probs = model.predict_proba(new_passengers)[:, 1]

print("1st class woman, 25, £80 fare -> P(survive) = {:.4f}".format(probs[0]))  # 0.9486
print("3rd class man,   30, £8  fare -> P(survive) = {:.4f}".format(probs[1]))  # 0.0976

# 94.9% versus 9.8%. The model has learned the two big patterns from Part 1
# and combined them into a single number per person.

1st class woman, 25, £80 fare -> P(survive) = 0.9486
3rd class man,   30, £8  fare -> P(survive) = 0.0976


---
# Where this goes next

- **Friday**: neural networks, and building a 1-neuron perceptron. That perceptron is startlingly close to the logistic regression you just built, which is a useful thing to notice.
- **Monday**: K-Means clustering, the unsupervised side of Part 2. Same Titanic passengers, but you throw away the `survived` column and see what groups fall out.
- **Next week**: model evaluation. Accuracy is a crude measure and it hides real problems, which is why Stage 6 is not the end of the story.

> ❗**Something to chew on before then.** Your model got about 80% accuracy. Suppose you'd been predicting a rare disease that affects 1 in 100 people. Predicting "healthy" for everybody would score 99%. Would that be a good model? That question is what next week is about.